# LLM

In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

from langchain.chat_models import init_chat_model

#Groq
groq_llm = init_chat_model(
    model="openai/gpt-oss-20b",
    model_provider="groq"
)
groq_llm.invoke("What is a cat?")

AIMessage(content='A **cat** (scientific name *Felis catus*) is a small, typically furry, carnivorous mammal that is one of the most common domestic pets worldwide. Here are some key points that capture what a cat is:\n\n| Feature | Details |\n|--------|---------|\n| **Taxonomy** | Kingdom: Animalia → Phylum: Chordata → Class: Mammalia → Order: Carnivora → Family: Felidae → Genus: *Felis* → Species: *Felis catus* |\n| **Physical traits** | • Body length 23–48\u202fcm (excluding tail) and a tail about the same length. <br>• Weight ranges from 3–8\u202fkg (average ~4–5\u202fkg). <br>• Sharp retractable claws, highly flexible spine, keen night vision, and excellent hearing. |\n| **Behavior** | • Solitary hunters, though many live in social groups when domesticated. <br>• Known for independent, sometimes aloof behavior but can form strong bonds with humans. <br>• Communicates via purring, hissing, growling, body language, and vocalizations. |\n| **Domestication** | • First domesticated aro

# Setting up the prompt (System Instructions)

In [6]:
prompt = (
        "You are a helpful assistant."
    )

# Setting up the agents

In [7]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
agent = create_agent(
    model=groq_llm,
    tools=[],
    system_prompt=prompt,
    # Remembers the conversation per thread_id (used by telegram_bot.py, one thread per chat)
    # checkpointer=InMemorySaver(),
)

agent.invoke({"messages": [{"role": "user", "content": "What is a cat"}]})


{'messages': [HumanMessage(content='What is a cat', additional_kwargs={}, response_metadata={}, id='d5e7d87f-568b-4d3b-adbc-ceb4bea7b18f'),
  AIMessage(content='A **cat** (scientific name *Felis catus*) is a small, typically furry, carnivorous mammal that is often kept as a pet. Cats are known for:\n\n- **Size**: Usually 2.5–4\u202fkg (5.5–9\u202flb) and about 46–51\u202fcm (18–20\u202fin) long, excluding the tail.\n- **Body**: Slender, flexible, with sharp retractable claws.\n- **Senses**: Excellent night vision, acute hearing, and a highly developed sense of smell.\n- **Behavior**: Independent, curious, and often affectionate. They hunt small prey (e.g., rodents, insects) and are adept climbers.\n- **Reproduction**: Females (queens) can have multiple litters each year, with 3–6 kittens per litter.\n- **Domestication**: Humans have kept cats for companionship and pest control for thousands of years.\n\nCats are beloved worldwide for their playful antics, soothing purring, and the comp

# PREPARING the ML PICKLE FILE

* Getting the DATA
* EDA
* Preprocessing before split
* Split
* Preprocessing after split
* Model

#### First we create our entire pipeline

In [9]:
import pandas as pd
df = pd.read_csv('kyphosis.csv')

#Preprocess before split
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df['Kyphosis'] = le.fit_transform(df['Kyphosis'])

#Split
from sklearn.model_selection import train_test_split

X = df.drop(columns=['Kyphosis'])
y = df['Kyphosis']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

#Preprocess after split
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
X_train[['Age']] = scaler.fit_transform(X_train[['Age']])
X_test[['Age']] = scaler.transform(X_test[['Age']])


#------------MODEL-----------------------------

from sklearn.tree import DecisionTreeClassifier
dt = DecisionTreeClassifier()
dt.fit(X_train,y_train)

preds = dt.predict(X_test)

from sklearn.metrics import classification_report

print(classification_report(preds,y_test))



              precision    recall  f1-score   support

           0       0.84      0.80      0.82        20
           1       0.33      0.40      0.36         5

    accuracy                           0.72        25
   macro avg       0.59      0.60      0.59        25
weighted avg       0.74      0.72      0.73        25



### Then we dump our pipeline into pickle files

In [16]:
dir(scaler)
le.m

array([-0.00515464])

In [ ]:
import pickle

preprocessing = {
    "age_scaler": scaler,
    "label_encoder": le,
}

with open("preprocessing.pkl", "wb") as f:
    pickle.dump(preprocessing, f)

with open("model.pkl", "wb") as f:
    pickle.dump(model, f)

# Testing the pickle files

In [ ]:
model = pickle.load(open("model.pkl", "rb"))
preprocessing = pickle.load(open("preprocessing.pkl", "rb"))

patient = pd.DataFrame({"Age": [15], "Number": [12], "Start": [7]})
patient[['Age']] = preprocessing["age_scaler"].transform(patient[['Age']])
prediction = model.predict(patient)
prediction

# Creating a tool for our agent that utilizes the ML pkl file

In [23]:
import pickle
def kyphosis_predict(age:int, number:int,start:int)->int:
    """This tool would be used to predict whether kyphosis is absent or present in a patient.
    It takes Age, Number and start values from user
    age: age of the child in MONTHS.
    number: number of vertebrae involved in the operation.
    start: number of the first vertebra operated on."""
    model = pickle.load(open("model.pkl", "rb"))
    preprocessing = pickle.load(open("preprocessing.pkl", "rb"))

    patient = pd.DataFrame({"Age": [15], "Number": [12], "Start": [7]})
    patient[['Age']] = preprocessing["age_scaler"].transform(patient[['Age']])
    prediction = model.predict(patient)
    
    return prediction

kyphosis_predict(12,5,7)
    

array([1])

# Integrating our kyphosis MOdel into our agent

In [20]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
agent = create_agent(
    model=groq_llm,
    tools=[kyphosis_predict],
    system_prompt="You are a helpful assistant, whenever user gives you info about their childs spinal surgery. Use that to predict whether kyphosis is absent or not, DO NOT do on your own, always use the predict_kyphosis functoin",
    # Remembers the conversation per thread_id (used by telegram_bot.py, one thread per chat)
    # checkpointer=InMemorySaver(),
)

agent.invoke({"messages": [{"role": "user", "content": "A child aged 71 months, 3 vertebrae involved, starting at vertebra 5. Will they get kyphosis?"}]})


{'messages': [HumanMessage(content='A child aged 71 months, 3 vertebrae involved, starting at vertebra 5. Will they get kyphosis?', additional_kwargs={}, response_metadata={}, id='14e9db32-acc2-45b0-8f97-a904a414db05'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'The user asks: "Will they get kyphosis?" We have instructions: "Use that to predict whether kyphosis is absent or not, DO NOT do on your own, always use the predict_kyphosis function." So we must call the function. The function expects age, number, start. The user gave age 71 months, number 3, start 5. So we call the function.', 'tool_calls': [{'id': 'fc_15d28434-7883-4a83-afff-c45bdf491b59', 'function': {'arguments': '{"age":71,"number":3,"start":5}', 'name': 'kyphosis_predict'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 120, 'prompt_tokens': 266, 'total_tokens': 386, 'completion_time': 0.133278281, 'completion_tokens_details': {'reasoning_tokens': 86}, 'prompt_time': 

# Implementing RAG
* First we do everything for preparing our vector DATABASE
* RAG officially starts with Retrieval that is search

In [30]:
DOCS_FOLDER = os.path.join("Files", "Docs")
INDEX_FOLDER = "files_index"

from langchain_community.document_loaders import PyPDFLoader, Docx2txtLoader, TextLoader
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

def load_docs():
    documents = []
    for file_name in os.listdir(DOCS_FOLDER):
        path = os.path.join(DOCS_FOLDER, file_name)

        if file_name.lower().endswith(".pdf"):
            loader = PyPDFLoader(path)



        elif file_name.lower().endswith(".docx"):
            loader = Docx2txtLoader(path)



        elif file_name.lower().endswith((".txt", ".md")):
            loader = TextLoader(path, encoding="utf-8")

            
        else:
            print("Skipped:", file_name)
            continue

        documents.extend(loader.load())
        print("Loaded doc:", file_name)
    return documents

# CHUNKING

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

def build_vector_db():
    documents = load_docs()

    splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)
    chunks = splitter.split_documents(documents)

    vector_db = FAISS.from_documents(chunks, embeddings)
    vector_db.save_local(INDEX_FOLDER)
    print(f"Saved {len(chunks)} chunks to {INDEX_FOLDER}")

build_vector_db()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loaded doc: Nestle doc.txt
Loaded doc: Nestle Process Document 2025 1.docx
Loaded doc: PN MarketCast Pixel .docx
Loaded doc: PN Marketccast doc.txt
Saved 23 chunks to files_index


In [ ]:
# Retrieval
vector_db = None
def search(question, k=4):
    global vector_db
    if vector_db is None:
        vector_db = FAISS.load_local(INDEX_FOLDER, embeddings, allow_dangerous_deserialization=True)
    return vector_db.similarity_search(question, k=k)
search("Metamedia")

[Document(id='266acfa1-20e9-42c4-a898-0f2b4e2af2d5', metadata={'source': 'Files\\Docs\\PN Marketccast doc.txt'}, page_content='Step 4: adding video creatives to the MarketCast folder\nAfter applying the pixels:\n1.\tGo to the creative folder\n2.\tSearch for the video asset assigned to the ads that you worked on.\n3.\tIf the video isn’t in the folder, please upload it \nThe same video can be used across multiple campaigns. If the video is already available, there’s no need to add it again.\nIf you are working on a creative swap where the pixel was already implemented, please make sure the new creative is available in the folder\n \n________________________________________\nStep 5: QA + document\nAfter applying:\n☐ Ad is Active\n☐ Pixel shows as applied at ad level\n☐ Note added to Jira confirming: \no\tPlacement tagged or excluded + reason\no\tCreative added to the creative folder'),
 Document(id='ab2936a0-3a10-4107-a966-4966a29976bf', metadata={'source': 'Files\\Docs\\PN Marketccast do

In [34]:
# Import the search function from your rag.py file
# from rag import search 

def search_private_documents(query: str) -> str:
    """Use this tool to search through the user's private documents, PDFs, text files, and images.
    Always use this tool if the user asks a question about their files, documents, or data.
    Input should be a clear search query."""
    
    try:
        #RETRIEVAL
        # Call the search function from rag.py
        results = search(query, k=4)
        
        if not results:
            return "No relevant information found in the documents."
        
        #AUGMENTATION
        # Format the retrieved chunks into a single string for the LLM to read
        formatted_results = []
        for i, doc in enumerate(results):
            source = doc.metadata.get("source", "Unknown file")
            content = doc.page_content
            formatted_results.append(f"--- Document {i+1} (Source: {source}) ---\n{content}")
            
        return "\n\n".join(formatted_results)
        
    except Exception as e:
        return f"An error occurred while searching the documents: {str(e)}"
search_private_documents('what website would be used for EMEA market in place of DV360 Reference from Nestle doc')

'--- Document 1 (Source: Files\\Docs\\Nestle doc.txt) ---\nNote:\n\n-\tFor Advertiser which have “Metamedia” as its subaccount and site given DV360 in T-sheet, Make sure to use “DV360 All Markets 9788961” site.\nFor other Advertiser which don’t have subaccount and site given as “DV360” use - DV360 All Markets 9697679\n-\tIn MENA markets if planners mention Shahid, watchit sites and mention as the placements run in DV360, make sure to use AODDBM site (Which is linked/associated with DV360 under its respective advertiser) and traffic as “Site served”. Take double confirmation on adserve method before trafficking.\n-\tAlways use Display advertisers, Except for Taiwan Market(In Taiwan market we have only parent advertisers)\n-\tPlease refer to the advertisers list document to find out the advertiser in case of any confusions.\n\n\nDocuments:\n\n--- Document 2 (Source: Files\\Docs\\Nestle Process Document 2025 1.docx) ---\nIn MENA markets if planners mention Shahid, watchit sites and mentio

In [35]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
agent = create_agent(
    model=groq_llm,
    tools=[kyphosis_predict,search_private_documents],
    system_prompt="You are a helpful assistant, whenever user gives you info about their childs spinal surgery. Use that to predict whether kyphosis is absent or not, DO NOT do on your own, always use the predict_kyphosis functoin.whenevr user asks about a question regarding documents, use search_private_documents to get the context for answering that question",
    # Remembers the conversation per thread_id (used by telegram_bot.py, one thread per chat)
    # checkpointer=InMemorySaver(),
)

agent.invoke({"messages": [{"role": "user", "content": "A child aged 71 months, 3 vertebrae involved, starting at vertebra 5. Will they get kyphosis?"}]})


{'messages': [HumanMessage(content='A child aged 71 months, 3 vertebrae involved, starting at vertebra 5. Will they get kyphosis?', additional_kwargs={}, response_metadata={}, id='6919db5f-f3f1-4b83-89e2-3be7538567cd'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'User asks about kyphosis prediction. We must use predict_kyphosis function. Provide call.', 'tool_calls': [{'id': 'fc_f3379bfc-6b0d-402a-b450-fba35bb56b33', 'function': {'arguments': '{"age":71,"number":3,"start":5}', 'name': 'kyphosis_predict'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 56, 'prompt_tokens': 351, 'total_tokens': 407, 'completion_time': 0.056547218, 'completion_tokens_details': {'reasoning_tokens': 22}, 'prompt_time': 0.020893892, 'prompt_tokens_details': None, 'queue_time': 0.336439406, 'total_time': 0.07744111}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_587d0d67cc', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logpro

In [36]:

agent.invoke({"messages": [{"role": "user", "content": "Go through the docs and tell me, what website would be used for EMEA market in place of DV360 Reference from Nestle doc and please asnwer in bullet piints, les than 50 words. Simple t ou understad"}]})


{'messages': [HumanMessage(content='Go through the docs and tell me, what website would be used for EMEA market in place of DV360 Reference from Nestle doc and please asnwer in bullet piints, les than 50 words. Simple t ou understad', additional_kwargs={}, response_metadata={}, id='62e56b94-13a3-4707-a00d-d788d9ddcd13'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'User asks: "Go through the docs and tell me, what website would be used for EMEA market in place of DV360 Reference from Nestle doc and please asnwer in bullet piints, les than 50 words. Simple t ou understad"\n\nWe need to use search_private_documents to find info in user\'s private documents. So we should call search_private_documents with query about Nestle doc DV360 Reference EMEA market website.', 'tool_calls': [{'id': 'fc_368c5f0e-afc0-482e-86f6-f9d5a6e8ad0f', 'function': {'arguments': '{"query":"Nestle doc DV360 Reference EMEA market website"}', 'name': 'search_private_documents'}, 'type': 'functio